In [71]:
import numpy as np
import mindspore as ms
from mindspore import context, nn, ops, Tensor, save_checkpoint
from mindspore.dataset import GeneratorDataset, transforms, vision
from mindspore.train import Model, LossMonitor, TimeMonitor
from mindspore.nn import CrossEntropyLoss, SGD, Accuracy
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import os
from PIL import Image

In [72]:
context.set_context(mode=context.GRAPH_MODE, device_target="CPU") 

ruta_train = 'C:/Users/migue/Desktop/UNI/Proyecto Huawei/NexuDream-Proyect/Nueva carpeta/train_dataset'
ruta_val = 'C:/Users/migue/Desktop/UNI/Proyecto Huawei/NexuDream-Proyect/Nueva carpeta/val_dataset'

[WARNING] ME(17772:45408,MainProcess):2026-02-27-12:02:09.479.000 [mindspore\context.py:1334] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.


In [80]:
class CustomImageGenerator:

    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.samples = []  
        self.class_names = {}  
        
        classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_names = {cls_name: idx for idx, cls_name in enumerate(classes)}
        self.idx_to_class = {idx: name for name, idx in self.class_names.items()}

        print(f"Clases detectadas: {len(self.class_names)}")
                
        for class_name, class_idx in self.class_names.items():
            class_dir = os.path.join(root_dir, class_name)
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp', '.JPG', '.PNG')):
                    img_path = os.path.join(class_dir, img_name)
                    self.samples.append((img_path, class_idx))
        
        print(f"Total de imágenes: {len(self.samples)}")
    
    def __getitem__(self, index):
        img_path, label = self.samples[index]
        img = Image.open(img_path).convert('RGB')
        return np.array(img), label
    
    def __len__(self):
        return len(self.samples)

In [81]:
train_transform = transforms.Compose([
    vision.Resize((256, 256)),
    vision.RandomHorizontalFlip(prob=0.5),
    vision.RandomRotation(degrees=30),
    vision.RandomColorAdjust(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2)),
    vision.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    vision.Rescale(1.0 / 255.0, 0.0),
    vision.Normalize(mean=[0.4598, 0.4815, 0.4021], std=[0.1915, 0.1677, 0.2100]),
    vision.HWC2CHW()
])

val_transform = transforms.Compose([
    vision.Resize((256, 256)),
    vision.Rescale(1.0 / 255.0, 0.0),
    vision.Normalize(mean=[0.4598, 0.4815, 0.4021], std=[0.1915, 0.1677, 0.2100]),
    vision.HWC2CHW()
])

In [82]:
train_generator = CustomImageGenerator(ruta_train)
val_generator = CustomImageGenerator(ruta_val)

class_names = train_generator.class_names
idx_to_class = train_generator.idx_to_class
num_classes = len(class_names)

train_dataset = GeneratorDataset(
    source=train_generator,
    column_names=["image", "label"],
    shuffle=True
)

val_dataset = GeneratorDataset(
    source=val_generator,
    column_names=["image", "label"],
    shuffle=False
)

train_dataset = train_dataset.map(operations=train_transform, input_columns=["image"])
val_dataset = val_dataset.map(operations=val_transform, input_columns=["image"])

train_dataset = train_dataset.batch(32, drop_remainder=True)
val_dataset = val_dataset.batch(32, drop_remainder=True)



Clases detectadas: 18
Total de imágenes: 21738
Clases detectadas: 18
Total de imágenes: 5441


In [83]:
class ResBlock(nn.Cell):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, pad_mode='pad', padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, pad_mode='pad', padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        
        self.shortcut = nn.SequentialCell()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.SequentialCell([
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, pad_mode='pad', padding=0),
                nn.BatchNorm2d(out_channels)
            ])
            
    def construct(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out

class ResNet(nn.Cell):
    def __init__(self, block, num_blocks, num_classes=18):
        super().__init__()
        self.in_channels = 32
        self.relu = nn.ReLU()
        
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, pad_mode='pad', padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        self.layer1 = self._make_layer(block, 32, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 64, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 128, num_blocks[2], stride=2)
        
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Dense(128, num_classes)
        self.dropout = nn.Dropout(p=0.5)
        
    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
        return nn.SequentialCell(layers)
        
    def construct(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avg_pool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


In [84]:
model = ResNet(ResBlock, [2, 2, 2], num_classes=num_classes)
loss = CrossEntropyLoss()
optimizer = SGD(model.trainable_params(), learning_rate=1e-2, momentum=0.9)

metrics = {'accuracy': Accuracy()}
ms_model = Model(model, loss_fn=loss, optimizer=optimizer, metrics=metrics)

In [ ]:
ms_model.fit(epoch=50, train_dataset=train_dataset, callbacks=[LossMonitor(), TimeMonitor()])

In [ ]:
save_checkpoint(model, 'model_deteccion_enfermedades_plantas.ckpt')

In [ ]:
model.set_train(False)

y_true = []
y_pred = []

for data in val_dataset.create_dict_iterator():
    imgs = data["image"]
    labels = data["label"]
    
    outputs = model(imgs)
    preds = ops.argmax(outputs, axis=1)
    
    y_pred.extend(preds.asnumpy().tolist())
    y_true.extend(labels.asnumpy().tolist())

sorted_class_names = [idx_to_class[i] for i in range(len(idx_to_class))]

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted_class_names)
fig, ax = plt.subplots(figsize=(15, 15))
disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation=90)
plt.title("Matriz de Confusión (MindSpore)")
plt.tight_layout()
plt.savefig("confusion_matrix_mindspore.png", dpi=300)
plt.show()

report = classification_report(y_true, y_pred, target_names=sorted_class_names, digits=3)
print("\nReporte de Clasificación:\n")
print(report)